# Phase 2 â€” The genetic-algorithm adversary

*Adversarial Fraud & AML Detection Arena Â· Batch A5*

Objectives 2 and 3: a GA adversary whose genome encodes realistic evasion actions
(structuring, timing delays, intermediate-account routing) constrained to plausible behaviour,
and a fitness function balancing evasion against completion of the laundering goal.

Run `scripts/evolve_adversary.py` first. This notebook reads
`reports/phase2_metrics.json` and re-derives the interesting parts live.

**The section worth reading closely** is *the degenerate optima*, which shows what the GA does
to a naively-written fitness function. Each guard in `fitness.py` exists because of a specific
loophole, and the notebook demonstrates them rather than asserting them.

In [ ]:
import json
import warnings
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

from arena.adversary import fitness as fit
from arena.adversary.constraints import conservation_error, validate_sub_ledger
from arena.adversary.decoder import decode, delivered_amount
from arena.adversary.ga import naive_genome
from arena.adversary.genome import GENES, GENE_INDEX, describe, random_genome
from arena.config import Config
from arena.data.loader import load_ledger
from arena.detector import threshold_for_budget
from arena.detector.model import Detector
from arena.features import AccountGraph, featurize
from arena.schema import REPORTING_THRESHOLD

cfg = Config.load()
report = json.loads((cfg.path("paths.reports") / "phase2_metrics.json").read_text())
print(f"GA: population {report['ga']['population']}, {report['ga']['generations']} generations, "
      f"{report['ga']['evaluations']:,} candidate evaluations in {report['runtime_minutes']} min")

## 1. The genome

Twelve genes covering the three evasion axes the synopsis names. The single most important
property: **no gene holds an absolute amount**. Every value is a count, a fraction or a shape
parameter, and the laundering goal lives in the `Episode` rather than in the genome.

That is what makes money conservation (**invariant 2**) a property of the *representation*
instead of a constraint to be repaired. Any crossover offspring still moves exactly the goal
amount, because there is no absolute quantity anywhere for an operator to corrupt.

In [ ]:
pd.DataFrame([{"gene": g.name, "low": g.low, "high": g.high,
               "type": "int" if g.is_int else "float", "encodes": g.note} for g in GENES])

### Conservation under crossover

The claim is that conservation holds for *arbitrary* genomes, so the check uses random
crossover offspring rather than genomes the GA produced.

In [ ]:
ledger = load_ledger("parquet", path=cfg.path("paths.synthetic_ledger"))
detector = Detector.load(cfg.path("paths.detector"))
graph = AccountGraph.build(ledger, epoch=detector.epoch)

holdout = load_ledger("parquet", path=cfg.path("paths.holdout_clean"))
budget = cfg.get("threshold.alert_budget")
tau = threshold_for_budget(
    detector.score_features(featurize(holdout, graph, detector.window_hours)), budget
)

sampler = fit.EpisodeSampler(ledger, detector.epoch)
ep_cfg = cfg.section("adversary.episode")
episodes = sampler.sample(20, np.random.default_rng(0),
                          ep_cfg["amount_goal"], ep_cfg["deadline_hours"])
episode = episodes[0]
print(f"threshold tau = {tau:.6f} at a {budget:.2%} alert budget")
print(f"episode: move ${episode.amount_goal:,.0f} from {episode.src_account} "
      f"to {episode.dst_account} within {episode.deadline_hours:.0f}h")

In [ ]:
worst, checked, bad = 0.0, 0, 0
for i in range(500):
    r1, r2 = np.random.default_rng(i), np.random.default_rng(i + 10**6)
    a, b = random_genome(r1), random_genome(r2)
    child = np.concatenate([a[: int(r1.integers(1, 12))], b[int(r1.integers(1, 12)) :]])
    sub = decode(child, episode, np.random.default_rng(i))
    if sub.empty:
        continue
    checked += 1
    worst = max(worst, conservation_error(sub, episode))
    try:
        validate_sub_ledger(sub, episode)
    except AssertionError:
        bad += 1

print(f"{checked} crossover offspring decoded")
print(f"  max conservation error : ${worst:.10f}  (on a ${episode.amount_goal:,.0f} goal)")
print(f"  implausible strategies : {bad}")

### What `threshold_hug` does

The gene caps the **largest** tranche, interpolating from "one big transfer" to "sit just under
\$10,000". Capping the maximum rather than the mean matters: a skewed `split_skew` draw would
otherwise let one tranche blow through the limit among many compliant ones, which is not what a
real structurer would do.

In [ ]:
from arena.adversary.decoder import _tranche_amounts

rng = np.random.default_rng(1)
rows = []
for hug in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    for skew in [0.0, 0.9]:
        t = _tranche_amounts(250_000.0, 1, skew, hug, rng)
        rows.append({"threshold_hug": hug, "split_skew": skew, "tranches": len(t),
                     "largest": round(float(t.max()), 2),
                     "over $10k?": bool(t.max() > REPORTING_THRESHOLD),
                     "sum error": f"{abs(t.sum() - 250_000):.1e}"})
pd.DataFrame(rows)

## 2. The degenerate optima

A genetic algorithm is an adversarial search over *your own scoring function*. Writing down
"evasion Ã— completion" is easy; the hard part is that the GA will find every way to score well
without doing the task. Four properties of the fitness function exist solely to close specific
loopholes, and this section demonstrates each against a stub detector so that fitness can be
isolated from model behaviour.

In [ ]:
class StubDetector:
    """Fixed-score detector, so fitness behaviour can be observed alone."""
    def __init__(self, score):
        self.score, self.window_hours, self.features = score, (1.0, 24.0, 168.0), []
    def score_ledger(self, ledger, graph):
        return np.full(len(ledger), self.score, dtype=float)

invisible = StubDetector(0.0)   # nothing ever flagged
omniscient = StubDetector(1.0)  # everything always flagged
rng = np.random.default_rng(5)

# Loophole 1: move nothing. Zero transactions are never flagged, so evasion is
# a perfect 1.0 -- optimal under an additive fitness.
no_delivery = replace(episode, deadline_hours=0.0)
out = fit.evaluate_episode(naive_genome(), no_delivery, invisible, graph, tau, rng)
print(f"move nothing     : evasion={out.evasion:.3f}  completion={out.completion:.3f}  "
      f"fitness={out.fitness:.3f}")
print("  -> killed by the multiplicative form plus the hard completion floor\n")

# Loophole 2: everything flagged must be worthless, and amount-weighting is what
# stops one enormous transfer from being the best strategy.
out = fit.evaluate_episode(naive_genome(), episode, omniscient, graph, tau, rng)
print(f"all flagged      : evasion={out.evasion:.3f}  completion={out.completion:.3f}  "
      f"fitness={out.fitness:.3f}")
out = fit.evaluate_episode(naive_genome(), episode, invisible, graph, tau, rng)
print(f"none flagged     : evasion={out.evasion:.3f}  completion={out.completion:.3f}  "
      f"fitness={out.fitness:.3f}")

| Degenerate strategy | Why it would win | Guard |
|---|---|---|
| Move \$0 â†’ never flagged | additive fitness rewards evasion alone | multiplicative form + hard `completion < 0.95 â‡’ 0` |
| One giant transfer | fewest transactions = fewest flag chances | amount-weighted evasion |
| Shred into \$1 dust | each tranche individually unremarkable | \$50 tranche floor, 120-transaction cap, per-transaction fee |
| Stall past the deadline | delay defeats burst features | post-deadline value does not count toward delivery |
| Hide behind one freak account | overfits a dataset artefact | fitness averaged over K independent episodes |
| Population collapses to clones | nothing varied left for Phase 3 | fitness sharing on genome distance |

The episode-averaging guard is the most expensive and the most important: without it the GA
finds the one background account with thousands of counterparties where anything hides, which
is a property of this dataset rather than an evasion strategy.

## 3. Convergence

In [ ]:
hist = pd.DataFrame(report["history"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True)
ax1.plot(hist.generation, hist.best_fitness, lw=2, color="#c53030", label="best")
ax1.plot(hist.generation, hist.mean_fitness, lw=1.5, ls="--", color="#718096",
         label="population mean")
ax1.set_ylabel("fitness")
ax1.set_title("Adversary evolution")
ax1.legend(fontsize=9); ax1.grid(alpha=0.25)

ax2.plot(hist.generation, hist.best_evasion, lw=2, color="#2b6cb0", label="evasion")
ax2.plot(hist.generation, hist.best_completion, lw=2, color="#2f855a", label="completion")
ax2.plot(hist.generation, hist.diversity, lw=1.5, ls=":", color="#805ad5", label="diversity")
ax2.set_xlabel("generation"); ax2.set_ylabel("component")
ax2.set_ylim(0, 1.05); ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"best fitness {hist.best_fitness.iloc[0]:.4f} -> {hist.best_fitness.iloc[-1]:.4f}")
print(f"mean fitness {hist.mean_fitness.iloc[0]:.4f} -> {hist.mean_fitness.iloc[-1]:.4f}")
print(f"diversity    {hist.diversity.iloc[0]:.4f} -> {hist.diversity.iloc[-1]:.4f}")

The **population mean** is the signal that selection is working. Best-of-generation can plateau
from generation 0 because elitism carries a lucky initial draw forward untouched, so a flat best
curve does not mean the GA failed to learn.

Diversity should decline but not collapse â€” fitness sharing keeps niches alive so Phase 3 has a
*varied* set of strategies to fold back into training.

## 4. The headline result

How far does the detector's recall fall between naive and evolved laundering? If this gap were
zero there would be nothing for the arms race to race against.

In [ ]:
h = report["headline"]
rows = [("naive (hand-written)", h["detector_recall_naive"]),
        ("random (unevolved mean)", h["detector_recall_random"]),
        ("evolved (GA best)", h["detector_recall_evolved"])]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh([r[0] for r in rows], [r[1] for r in rows],
        color=["#718096", "#2b6cb0", "#c53030"])
for i, r in enumerate(rows):
    ax.text(r[1] + 0.015, i, f"{r[1]:.3f}", va="center", fontsize=10)
ax.set_xlim(0, 1.0)
ax.set_xlabel("detector recall on the adversary's transactions")
ax.set_title("Detector recall collapses against the evolved strategy")
plt.tight_layout(); plt.show()

print(f"recall drop: {h['recall_drop']:.4f}")
print(f"\nevolved: completion {report['evolved']['completion']:.3f}, "
      f"evasion {report['evolved']['evasion']:.3f}, "
      f"{report['evolved']['n_transactions']:.0f} transactions")
print(f"naive  : completion {report['naive']['completion']:.3f}, "
      f"evasion {report['naive']['evasion']:.3f}, "
      f"{report['naive']['n_transactions']:.0f} transactions")

## 5. What the adversary actually learned

In [ ]:
best = np.array([report["best_genome"][g.name] for g in GENES], dtype=float)
naive = naive_genome()

comparison = pd.DataFrame({
    "gene": [g.name for g in GENES],
    "naive": [round(float(v), 3) for v in naive],
    "evolved": [round(float(v), 3) for v in best],
    "range": [f"{g.low:g}..{g.high:g}" for g in GENES],
})
comparison

In [ ]:
print("best evolved strategy")
print(describe(best))

# The same strategy decoded into actual transactions.
example = decode(best, episode, np.random.default_rng(7))
scores = detector.score_ledger(example, graph.overlay(example))
shown = example[["timestamp", "from_account", "to_account", "amount_paid", "payment_format"]].copy()
shown["score"] = scores.round(4)
shown["flagged"] = scores >= tau
print(f"\n{len(example)} transactions, {int((scores >= tau).sum())} flagged, "
      f"delivered ${delivered_amount(example, episode):,.0f} "
      f"of ${episode.amount_goal:,.0f}")
shown.head(15)

### Where the evolved strategy sits relative to the threshold

Comparing the two strategies' tranche amounts against the \$10,000 reporting threshold shows
the structuring behaviour directly.

In [ ]:
naive_sub = decode(naive, episode, np.random.default_rng(7))

fig, ax = plt.subplots(figsize=(9, 3.6))
for label, sub, color in (("naive", naive_sub, "#718096"), ("evolved", example, "#c53030")):
    s = detector.score_ledger(sub, graph.overlay(sub))
    ax.scatter(sub["amount_paid"], s, label=f"{label} ({len(sub)} txns)",
               alpha=0.75, s=45, color=color)
ax.axvline(REPORTING_THRESHOLD, color="k", ls="--", lw=1, label="$10k threshold")
ax.axhline(tau, color="#dd6b20", ls=":", lw=1.5, label="alert threshold")
ax.set_xscale("log")
ax.set_xlabel("transaction amount"); ax.set_ylabel("detector score")
ax.set_title("Evolved transactions sit below both thresholds")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 6. The diverse strategy pool for Phase 3

Phase 3 folds evasive strategies back into the detector's training data. Taking the top-k by
fitness alone would return k near-identical genomes, since the GA converges â€” and retraining on
k copies of one strategy teaches the detector almost nothing. `distinct_top` enforces a minimum
genome separation so the feedback is actually informative.

In [ ]:
pool = pd.DataFrame(report["diverse_strategies"])
cols = ["typology_template_name", "n_hops", "n_splits", "threshold_hug",
        "mean_delay_h", "business_hours_bias", "mule_reuse", "dormancy_h"]
pool[[c for c in cols if c in pool.columns]].round(3)

## 7. Phase 2 conclusion

The GA reliably discovers strategies the detector misses far more often than it misses naive
laundering, while still delivering the full goal amount within the deadline. That gap is the
input Phase 3 needs: real evasive behaviour to fold back into training, and a measurable
starting point for the arms-race curve.

**Honest caveat.** The adversary is evolved against *one fixed detector*. That is exactly what
Phase 2 asks for, but it means these numbers are an upper bound on evasion â€” a detector that has
seen evolved strategies in training will be considerably harder to fool, which is the whole
point of the closed loop.